# 🤖 Interface de Chat com Qwen

**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Autor:** Jader Lugon Junior

---

## 📋 Sobre este Notebook

Este notebook fornece uma **interface interativa** para conversar com o Qwen sobre conceitos de Fenômenos de Transporte. Ele é útil para:

- 📚 **Estudantes:** Tirar dúvidas sobre conceitos do livro
- 👨‍🏫 **Professores:** Preparar aulas e exercícios
- 🔬 **Pesquisadores:** Explorar ideias e gerar código

### ⚠️ IA Crítica — Lembre-se!

Conforme discutido no Capítulo 1 do livro, **sempre audite as respostas da IA**:

- ✅ Verifique consistência dimensional
- ✅ Confirme valores numéricos com cálculos independentes
- ✅ Valide interpretações físicas
- ✅ Compare com o conteúdo do livro

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os
from pathlib import Path

# Adicionar pasta raiz ao path
sys.path.insert(0, str(Path().resolve().parent))

# Importar módulos
from qwen_integration.qwen_client import QwenClient, get_client

# Widgets para interface
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print('⚠️  ipywidgets não instalado. Interface básica será usada.')
    print('   Instale com: pip install ipywidgets')

print('✅ Ambiente configurado')

---

## 🔑 1. Configuração da API Key

Antes de usar, configure sua chave de API do Qwen (DashScope).

In [ ]:
# ============================================================
# CONFIGURAÇÃO DA API KEY
# ============================================================

# Opção 1: Usar variável de ambiente (recomendado)
api_key = os.getenv('QWEN_API_KEY')

# Opção 2: Inserir manualmente (apenas para desenvolvimento)
if not api_key:
    if HAS_WIDGETS:
        print('⚠️  Variável QWEN_API_KEY não definida.')
        print('   Defina-a ou insira abaixo:')
        api_key_input = widgets.Password(
            description='API Key:',
            placeholder='Cole sua chave aqui'
        )
        display(api_key_input)
    else:
        api_key = input('Insira sua API Key: ')

print('✅ Configuração concluída')

---

## 🚀 2. Inicializar o Cliente

In [ ]:
# ============================================================
# INICIALIZAR CLIENTE
# ============================================================

if HAS_WIDGETS and 'api_key_input' in locals():
    api_key = api_key_input.value

try:
    client = QwenClient(
        api_key=api_key,
        modelo='qwen3.7',
        temperatura=0.7
    )
    print(f'✅ Cliente inicializado: {client}')
except Exception as e:
    print(f'❌ Erro ao inicializar: {e}')

---

## 💬 3. Interface de Chat Interativa

Use os widgets abaixo para conversar com o Qwen.

In [ ]:
# ============================================================
# INTERFACE DE CHAT
# ============================================================

if not HAS_WIDGETS:
    print('⚠️  Interface interativa requer ipywidgets.')
    print('   Use os métodos diretos do cliente (seção 4).')
else:
    # Widgets
    modo_select = widgets.Dropdown(
        options=[
            ('💬 Conversa Livre', 'livre'),
            ('📚 Explicar Conceito', 'explicar'),
            ('✏️ Resolver Exercício', 'resolver'),
            ('💻 Gerar Código', 'codigo')
        ],
        value='livre',
        description='Modo:',
        style={'description_width': 'initial'}
    )
    
    capitulo_select = widgets.Dropdown(
        options=[(f'Capítulo {i}', i) for i in range(1, 12)] + [('Não especificado', None)],
        value=None,
        description='Capítulo:',
        style={'description_width': 'initial'}
    )
    
    nivel_select = widgets.RadioButtons(
        options=['Graduação', 'Pós-Graduação'],
        value='Graduação',
        description='Nível:',
        style={'description_width': 'initial'}
    )
    
    mensagem_input = widgets.Textarea(
        placeholder='Digite sua pergunta, conceito ou exercício...',
        rows=4,
        layout=widgets.Layout(width='100%')
    )
    
    enviar_button = widgets.Button(
        description='📤 Enviar',
        button_style='primary',
        icon='paper-plane'
    )
    
    limpar_button = widgets.Button(
        description='🗑️ Limpar',
        button_style='warning'
    )
    
    saida_output = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #ccc',
            padding='10px',
            min_height='300px',
            overflow='auto'
        )
    )
    
    # Função de envio
    def on_enviar_clicked(b):
        with saida_output:
            clear_output()
            
            mensagem = mensagem_input.value.strip()
            if not mensagem:
                display(Markdown('⚠️ **Digite uma mensagem antes de enviar.**'))
                return
            
            modo = modo_select.value
            capitulo = capitulo_select.value
            nivel = nivel_select.value.lower().replace('-', '_')
            
            display(Markdown(f'### 🤖 Respondendo (modo: {modo})...'))
            
            try:
                if modo == 'livre':
                    resposta = client.conversar(mensagem)
                elif modo == 'explicar':
                    resposta = client.explicar_conceito(
                        conceito=mensagem,
                        capitulo=capitulo,
                        nivel=nivel
                    )
                elif modo == 'resolver':
                    resposta = client.resolver_exercicio(
                        enunciado=mensagem,
                        capitulo=capitulo
                    )
                elif modo == 'codigo':
                    resposta = client.gerar_codigo(
                        descricao=mensagem,
                        capitulo=capitulo
                    )
                
                clear_output()
                display(Markdown('---'))
                display(Markdown('### 💬 Sua pergunta:'))
                display(Markdown(f'> {mensagem}'))
                display(Markdown('---'))
                display(Markdown('### 🤖 Resposta do Qwen:'))
                display(Markdown(resposta.content))
                display(Markdown('---'))
                display(Markdown(
                    f'*Modelo: {resposta.modelo} | '
                    f'Tokens: {resposta.tokens_usados.get("total_tokens", "?")} | '
                    f'Tempo: {resposta.tempo_resposta:.2f}s*'
                ))
                
            except Exception as e:
                clear_output()
                display(Markdown(f'❌ **Erro:** {e}'))
    
    def on_limpar_clicked(b):
        client.limpar_historico()
        mensagem_input.value = ''
        with saida_output:
            clear_output()
            display(Markdown('🗑️ **Conversa limpa.**'))
    
    enviar_button.on_click(on_enviar_clicked)
    limpar_button.on_click(on_limpar_clicked)
    
    # Layout
    config_box = widgets.VBox([
        widgets.HBox([modo_select, capitulo_select]),
        nivel_select
    ])
    
    input_box = widgets.VBox([
        widgets.HTML('<b>Digite sua mensagem:</b>'),
        mensagem_input,
        widgets.HBox([enviar_button, limpar_button])
    ])
    
    display(widgets.VBox([
        widgets.HTML('<h3>⚙️ Configurações</h3>'),
        config_box,
        widgets.HTML('<hr>'),
        input_box,
        widgets.HTML('<hr>'),
        widgets.HTML('<h3>💬 Conversa</h3>'),
        saida_output
    ]))

---

## 🛠️ 4. Uso Direto (Sem Widgets)

Se preferir, use os métodos diretamente:

In [ ]:
# ============================================================
# EXEMPLOS DE USO DIRETO
# ============================================================

# Exemplo 1: Conversa livre
print('=== Exemplo 1: Conversa Livre ===')
resposta = client.perguntar('O que é o número de Reynolds?')
print(resposta.content[:200] + '...')
print(f'\nTokens: {resposta.tokens_usados}\n')

# Exemplo 2: Explicar conceito
print('=== Exemplo 2: Explicar Conceito ===')
resposta = client.explicar_conceito(
    conceito='Lei de Darcy-Weisbach',
    capitulo=4,
    nivel='graduacao'
)
print(resposta.content[:300] + '...')

# Exemplo 3: Resolver exercício
print('\n=== Exemplo 3: Resolver Exercício ===')
exercicio = '''
Calcule a perda de carga em um tubo de aço (ε = 0,15 mm) com
diâmetro D = 75 mm, comprimento L = 80 m, transportando água
(ν = 10⁻⁶ m²/s) com vazão Q = 3,0 L/s.
'''
resposta = client.resolver_exercicio(
    enunciado=exercicio,
    capitulo=4
)
print(resposta.content[:300] + '...')

# Exemplo 4: Gerar código
print('\n=== Exemplo 4: Gerar Código ===')
descricao = 'Função para calcular a vazão pela equação de Manning'
resposta = client.gerar_codigo(
    descricao=descricao,
    capitulo=5
)
print(resposta.content[:300] + '...')

---

## 💾 5. Salvar e Carregar Histórico

In [ ]:
# ============================================================
# SALVAR E CARREGAR HISTÓRICO
# ============================================================

# Salvar
client.salvar_historico('historico_chat.json')
print('✅ Histórico salvo em historico_chat.json')

# Limpar
client.limpar_historico()
print(f'🗑️ Histórico limpo ({len(client.historico)} mensagens)')

# Carregar
client.carregar_historico('historico_chat.json')
print(f'📥 Histórico carregado ({len(client.historico)} mensagens)')

---

## 🎓 6. Exemplos Pedagógicos

### 6.1 Preparando uma Aula sobre Trocadores de Calor

In [ ]:
# ============================================================
# EXEMPLO: PREPARAR AULA
# ============================================================

prompt_aula = '''
Estou preparando uma aula sobre Trocadores de Calor (Capítulo 9).
Quero explicar o método LMTD para alunos de graduação.

Por favor:
1. Explique o conceito de LMTD de forma intuitiva
2. Apresente a fórmula e explique cada termo
3. Dê um exemplo numérico simples
4. Sugira 3 perguntas para discutir em sala
'''

resposta = client.perguntar(prompt_aula)
from IPython.display import display, Markdown
display(Markdown(resposta.content))

### 6.2 Auditoria Crítica de Resposta da IA

In [ ]:
# ============================================================
# EXEMPLO: AUDITORIA CRÍTICA
# ============================================================

prompt_auditoria = '''
A IA respondeu que a perda de carga em um tubo de 100 m com
D = 50 mm e Q = 5 L/s é de 0,5 m. Isso está correto?

Verifique:
1. Consistência dimensional
2. Ordem de grandeza esperada
3. Cálculo independente usando Darcy-Weisbach
'''

resposta = client.perguntar(prompt_auditoria)
display(Markdown(resposta.content))

---

## 🔄 Navegação

- [🏠 Repositório Principal](../README.md)
- [📚 Notebooks dos Capítulos](../notebooks/)
- [🔬 Estudos de Caso](../casos/READMEcasos.md)
- [🤖 Documentação da Integração](README.md)

---

<div align="center">

**🎓 Use a IA com responsabilidade e pensamento crítico!**

*Lembre-se: a IA é uma ferramenta, não um substituto para o entendimento.*

</div>